#### Simple Gen AI APP Using Langchain

In [71]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ['OPENAI_API_KEY']=os.getenv("OPENAI_API_KEY")
## Langsmith Tracking
os.environ["LANGCHAIN_API_KEY"]=os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"]="true"
os.environ["LANGCHAIN_PROJECT"]=os.getenv("LANGCHAIN_PROJECT")

**Libraries Imported:**
- `os`: Used for interacting with the operating system and setting environment variables
- `dotenv`: Loads environment variables from a .env file into the application

**Purpose:** This cell sets up authentication and tracking for OpenAI API and LangSmith. The environment variables enable secure API access and allow LangSmith to trace and monitor your LangChain application's execution.

In [72]:
## Data Ingestion--From the website we need to scrape the data

from langchain_community.document_loaders import WebBaseLoader

**Library Imported:**
- `WebBaseLoader`: A document loader from LangChain Community that scrapes and loads content from web pages

**Purpose:** This loader enables you to extract text content from URLs and convert them into LangChain Document objects for further processing.

In [73]:
loader=WebBaseLoader("https://reference.langchain.com/python/langchain/agents/")
loader

**Function Used:**
- `WebBaseLoader(url)`: Creates a loader instance configured to scrape the specified URL

**Purpose:** Instantiates a WebBaseLoader object targeting the LangChain agents documentation page, preparing it for data extraction.

In [74]:
docs=loader.load()
docs

[Document(metadata={'source': 'https://reference.langchain.com/python/langchain/agents/', 'title': 'Agents | LangChain Reference', 'description': 'Unified reference documentation for LangChain and LangGraph Python packages.', 'language': 'en'}, page_content='\n\n\n\n\n\n\n\n\n\n\n\n\nAgents | LangChain Reference\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n          Skip to content\n        \n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n            LangChain Reference\n          \n\n\n\n            \n              Agents\n            \n          \n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n            Initializing search\n          \n\n\n\n\n\n\n\n\n\n\n\n\n    langchain-ai/docs\n  \n\n\n\n\n\n\n\n\n\n\n        \n  \n  \n    \n  \n  Get started\n\n      \n\n\n\n          \n  \n  \n    \n  \n  LangChain\n\n        \n\n\n\n          \n  \n  \n    \n  \n  LangGraph\n\n        \n\n\n\n          \n  \n  \n    \n  \n  Deep Agents\n\n        \n\n\n\n   

In [75]:
### Load Data--> Docs-->Divide our Docuemnts into chunks dcouments-->text-->vectors-->Vector Embeddings--->Vector Store DB
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
documents=text_splitter.split_documents(docs)
len(documents)

14

**Library Imported:**
- `RecursiveCharacterTextSplitter`: A text splitter that recursively divides documents using different separators (paragraphs, sentences, words) to create optimal chunks

**Functions Used:**
- `RecursiveCharacterTextSplitter(chunk_size, chunk_overlap)`: Creates a splitter with specified chunk size and overlap
- `split_documents()`: Splits the loaded documents into smaller chunks

**Purpose:** Breaks down large documents into smaller, manageable chunks of 1000 characters with 200 character overlap. This is crucial for embedding generation and retrieval, as it ensures each chunk fits within model token limits while maintaining context continuity.

**Purpose:** Displays the list of document chunks to inspect their content, metadata, and structure after splitting.

In [76]:
documents

[Document(metadata={'source': 'https://reference.langchain.com/python/langchain/agents/', 'title': 'Agents | LangChain Reference', 'description': 'Unified reference documentation for LangChain and LangGraph Python packages.', 'language': 'en'}, page_content='Agents | LangChain Reference\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n          Skip to content\n        \n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n            LangChain Reference\n          \n\n\n\n            \n              Agents\n            \n          \n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n            Initializing search\n          \n\n\n\n\n\n\n\n\n\n\n\n\n    langchain-ai/docs\n  \n\n\n\n\n\n\n\n\n\n\n        \n  \n  \n    \n  \n  Get started\n\n      \n\n\n\n          \n  \n  \n    \n  \n  LangChain\n\n        \n\n\n\n          \n  \n  \n    \n  \n  LangGraph\n\n        \n\n\n\n          \n  \n  \n    \n  \n  Deep Agents\n\n        \n\n\n\n          \n  \n  \n    \n  \

In [77]:
from langchain_openai import OpenAIEmbeddings
embeddings=OpenAIEmbeddings()


"**Library Imported:**
- `OpenAIEmbeddings`: LangChain wrapper for OpenAI's embedding models that converts text into vector representations

**Purpose:** Creates an embeddings object using OpenAI's text-embedding model (default: text-embedding-ada-002). Embeddings are numerical vector representations of text that capture semantic meaning, enabling similarity search and retrieval operations.

In [78]:
from langchain_community.vectorstores import FAISS
vectorstoredb=FAISS.from_documents(documents,embeddings)

**Library Imported:**
- `FAISS`: Facebook AI Similarity Search - a library for efficient similarity search and clustering of dense vectors

**Function Used:**
- `FAISS.from_documents(documents, embeddings)`: Creates a vector store from documents and generates embeddings for each chunk

**Purpose:** Builds an in-memory vector database by converting all document chunks into embeddings and storing them in a FAISS index. This enables fast similarity search for retrieval augmented generation (RAG).

In [79]:
vectorstoredb

In [80]:
## Query From a vector db

query="Give me the function for create agent"
result=vectorstoredb.similarity_search(query)
result[0].page_content

'agents\n¶\n\nEntrypoint to building Agents with LangChain.\n\n\n\n create_agent\n¶\ncreate_agent(\n    model: str | BaseChatModel,\n    tools: Sequence[BaseTool | Callable | dict[str, Any]] | None = None,\n    *,\n    system_prompt: str | SystemMessage | None = None,\n    middleware: Sequence[AgentMiddleware[StateT_co, ContextT]] = (),\n    response_format: ResponseFormat[ResponseT] | type[ResponseT] | None = None,\n    state_schema: type[AgentState[ResponseT]] | None = None,\n    context_schema: type[ContextT] | None = None,\n    checkpointer: Checkpointer | None = None,\n    store: BaseStore | None = None,\n    interrupt_before: list[str] | None = None,\n    interrupt_after: list[str] | None = None,\n    debug: bool = False,\n    name: str | None = None,\n    cache: BaseCache | None = None,\n) -> CompiledStateGraph[\n    AgentState[ResponseT], ContextT, _InputAgentState, _OutputAgentState[ResponseT]\n]'

**Function Used:**
- `similarity_search(query)`: Searches the vector store for documents most similar to the query string

**Purpose:** Performs semantic search to find the most relevant document chunks. The query is converted to an embedding and compared against stored embeddings using cosine similarity or distance metrics. Returns the top matching documents.

In [81]:
from langchain_openai import ChatOpenAI
llm=ChatOpenAI(model="gpt-4o")

**Library Imported:**
- `ChatOpenAI`: LangChain wrapper for OpenAI's chat models (GPT-3.5, GPT-4, etc.)

**Purpose:** Initializes a connection to OpenAI's GPT-4o model, which will be used to generate responses based on retrieved context. This is the language model that powers the conversational AI capabilities.

In [82]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_template(
    """
Answer the following question based only on the provided context.

Question: {input}

<context>
{context}
</context>
"""
)

document_chain = prompt | llm | StrOutputParser()


**Libraries Imported:**
- `ChatPromptTemplate`: Creates structured prompt templates with variables for dynamic input
- `StrOutputParser`: Parses LLM output and returns it as a plain string

**Functions Used:**
- `ChatPromptTemplate.from_template()`: Creates a prompt template with placeholders for `input` and `context`
- Pipe operator (`|`): Chains components together in a sequential pipeline

**Purpose:** Defines a prompt template that instructs the LLM to answer questions using only provided context. The document_chain creates a processing pipeline: prompt → LLM → string output. This ensures structured, context-aware responses.

In [83]:
from langchain_core.documents import Document
document_chain.invoke({
    "input":"Give me the function for create agent",
    "context":[Document(page_content="""Entrypoint to building Agents with LangChain.

 create_agent ¶


create_agent(
    model: str | BaseChatModel,
    tools: Sequence[BaseTool | Callable | dict[str, Any]] | None = None,
    *,
    system_prompt: str | SystemMessage | None = None,
    middleware: Sequence[AgentMiddleware[StateT_co, ContextT]] = (),
    response_format: ResponseFormat[ResponseT] | type[ResponseT] | None = None,
    state_schema: type[AgentState[ResponseT]] | None = None,
    context_schema: type[ContextT] | None = None,
    checkpointer: Checkpointer | None = None,
    store: BaseStore | None = None,
    interrupt_before: list[str] | None = None,
    interrupt_after: list[str] | None = None,
    debug: bool = False,
    name: str | None = None,
    cache: BaseCache | None = None,
) -> CompiledStateGraph[
    AgentState[ResponseT], ContextT, _InputAgentState, _OutputAgentState[ResponseT]
]
Creates an agent graph that calls tools in a loop until a stopping condition is met.

For more details on using create_agent, visit the Agents docs.

""")]
})

'The function for creating an agent is:\n\n```python\ncreate_agent(\n    model: str | BaseChatModel,\n    tools: Sequence[BaseTool | Callable | dict[str, Any]] | None = None,\n    *,\n    system_prompt: str | SystemMessage | None = None,\n    middleware: Sequence[AgentMiddleware[StateT_co, ContextT]] = (),\n    response_format: ResponseFormat[ResponseT] | type[ResponseT] | None = None,\n    state_schema: type[AgentState[ResponseT]] | None = None,\n    context_schema: type[ContextT] | None = None,\n    checkpointer: Checkpointer | None = None,\n    store: BaseStore | None = None,\n    interrupt_before: list[str] | None = None,\n    interrupt_after: list[str] | None = None,\n    debug: bool = False,\n    name: str | None = None,\n    cache: BaseCache | None = None,\n) -> CompiledStateGraph[\n    AgentState[ResponseT], ContextT, _InputAgentState, _OutputAgentState[ResponseT]\n]\n```'

**Library Imported:**
- `Document`: LangChain's document class that wraps text content with metadata

**Function Used:**
- `document_chain.invoke()`: Executes the chain with provided input and context

**Purpose:** Tests the document chain by manually providing a question and context as a Document object. This demonstrates how the chain processes inputs before integrating with the retriever.

However, we want the documents to first come from the retriever we just set up. That way, we can use the retriever to dynamically select the most relevant documents and pass those in for a given question.

In [84]:
### Input--->Retriever--->vectorstoredb

vectorstoredb

In [85]:
retriever=vectorstoredb.as_retriever()
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

# Create a proper document chain for retrieval
document_chain = create_stuff_documents_chain(llm, prompt)
retrieval_chain=create_retrieval_chain(retriever,document_chain)


**Functions/Classes Used:**
- `as_retriever()`: Converts the vector store into a retriever interface for document lookup
- `create_stuff_documents_chain()`: Creates a chain that combines ("stuffs") all retrieved documents into the prompt context
- `create_retrieval_chain()`: Combines the retriever and document chain into a complete RAG pipeline

**Purpose:** Builds a complete Retrieval Augmented Generation (RAG) system. When invoked with a query, the retrieval_chain automatically:
1. Retrieves relevant documents from the vector store
2. Passes them as context to the LLM
3. Generates an answer based on the retrieved information

In [86]:
retrieval_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x1423f39a0>, search_kwargs={}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='\nAnswer the following question based only on the provided context.\n\nQuestion: {input}\n\n<context>\n{context}\n</context>\n'), addition

In [87]:
## Get the response form the LLM
response=retrieval_chain.invoke({"input":"Give me the function for create agent"})
response['answer']

'The function for creating an agent is `create_agent`, and it has the following signature:\n\n```python\ncreate_agent(\n    model: str | BaseChatModel,\n    tools: Sequence[BaseTool | Callable | dict[str, Any]] | None = None,\n    *,\n    system_prompt: str | SystemMessage | None = None,\n    middleware: Sequence[AgentMiddleware[StateT_co, ContextT]] = (),\n    response_format: ResponseFormat[ResponseT] | type[ResponseT] | None = None,\n    state_schema: type[AgentState[ResponseT]] | None = None,\n    context_schema: type[ContextT] | None = None,\n    checkpointer: Checkpointer | None = None,\n    store: BaseStore | None = None,\n    interrupt_before: list[str] | None = None,\n    interrupt_after: list[str] | None = None,\n    debug: bool = False,\n    name: str | None = None,\n    cache: BaseCache | None = None,\n) -> CompiledStateGraph[\n    AgentState[ResponseT], ContextT, _InputAgentState, _OutputAgentState[ResponseT]\n]\n```'

**Function Used:**
- `retrieval_chain.invoke()`: Executes the full RAG pipeline with a user query

**Purpose:** Invokes the complete retrieval chain with a question about creating agents. The chain retrieves relevant documentation chunks, provides them as context to GPT-4o, and generates an answer. The response dictionary contains the 'answer' key with the LLM's response and 'context' with retrieved documents.

In [88]:
response

{'input': 'Give me the function for create agent',
 'context': [Document(id='82a389da-e1e7-4938-9eba-c65eb1fdad91', metadata={'source': 'https://reference.langchain.com/python/langchain/agents/', 'title': 'Agents | LangChain Reference', 'description': 'Unified reference documentation for LangChain and LangGraph Python packages.', 'language': 'en'}, page_content='agents\n¶\n\nEntrypoint to building Agents with LangChain.\n\n\n\n create_agent\n¶\ncreate_agent(\n    model: str | BaseChatModel,\n    tools: Sequence[BaseTool | Callable | dict[str, Any]] | None = None,\n    *,\n    system_prompt: str | SystemMessage | None = None,\n    middleware: Sequence[AgentMiddleware[StateT_co, ContextT]] = (),\n    response_format: ResponseFormat[ResponseT] | type[ResponseT] | None = None,\n    state_schema: type[AgentState[ResponseT]] | None = None,\n    context_schema: type[ContextT] | None = None,\n    checkpointer: Checkpointer | None = None,\n    store: BaseStore | None = None,\n    interrupt_befo

In [89]:
response['context']

[Document(id='82a389da-e1e7-4938-9eba-c65eb1fdad91', metadata={'source': 'https://reference.langchain.com/python/langchain/agents/', 'title': 'Agents | LangChain Reference', 'description': 'Unified reference documentation for LangChain and LangGraph Python packages.', 'language': 'en'}, page_content='agents\n¶\n\nEntrypoint to building Agents with LangChain.\n\n\n\n create_agent\n¶\ncreate_agent(\n    model: str | BaseChatModel,\n    tools: Sequence[BaseTool | Callable | dict[str, Any]] | None = None,\n    *,\n    system_prompt: str | SystemMessage | None = None,\n    middleware: Sequence[AgentMiddleware[StateT_co, ContextT]] = (),\n    response_format: ResponseFormat[ResponseT] | type[ResponseT] | None = None,\n    state_schema: type[AgentState[ResponseT]] | None = None,\n    context_schema: type[ContextT] | None = None,\n    checkpointer: Checkpointer | None = None,\n    store: BaseStore | None = None,\n    interrupt_before: list[str] | None = None,\n    interrupt_after: list[str] | 